Data Preparation

In [1]:
import pandas as pd 
import re

In [2]:
# load data 
df = pd.read_csv("cleaned_data/bangalore_cleaned.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170,2.0,1.0,38.00
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785,5.0,3.0,295.00


In [3]:
df.isnull().sum()

area_type       0
availability    0
location        0
size            0
society         0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


size is object type and total_sqft is also object type.

We will convert total_sqft to numeric and for different formats present in this feature eg 2100 - 2200 , we will take the average of the two numbers and convert the entire column to numeric type.

We will Extract BHK from the size column and this feature also has two unique formats eg. 2 BHK , 2 Bedroom. We will consider both of them as 2 BHK and convert the entire column to numeric type.

In [5]:
# function to convert the total_sqft column to numeric

def convert_sqft(x):
    try:
        s = str(x).strip()
        if "-" in s:
            parts = s.split("-")
            return (float(parts[0]) + float(parts[1]))/2
        return float(s)

    except: # if parsing fails, return None
        return None

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [6]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft      15
bath             0
balcony          0
price            0
dtype: int64

In [7]:
# Dropping the rows with null values in total_sqft column
df = df.dropna(subset=["total_sqft"])

In [8]:
# Function to extract BHK from the size column

def extract_bhk(x):
    if pd.isna(x): return None
    m = re.search(r"(\d+)\s*(bhk|bedroom)" , str(x).lower())

    if m:
        return int(m.group(1)) # return the number part as integer

    return None


df['bhk'] = df['size'].apply(extract_bhk)

In [9]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00,4.0
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00,3.0
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00,2.0
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785.0,5.0,3.0,295.00,4.0


In [10]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft       0
bath             0
balcony          0
price            0
bhk             10
dtype: int64

In [11]:
# drop the rows with null values in bhk column
df = df.dropna(subset=["bhk"])

In [12]:
# drop size column as we have extracted bhk from it

df = df.drop("size" , axis=1)

In [13]:
# Create a new column to provide descriptive text for each property listing

df['text'] = (
    df['bhk'].astype(str) + " BHK property in " + df['location'] +
    " with " + df['total_sqft'].astype(str) + " sqft, priced at " +
    df['price'].astype(str) + " lakh. " +
    "Bathrooms: " + df['bath'].astype(int).astype(str) + ". " +
    "Balconies: " + df['balcony'].astype(int).astype(str) + ". " +
    "Area type: " + df['area_type'] + ". " +
    "Availability: " + df['availability'] + ". " +
    "Society: " + df['society']
)

In [14]:
df.head()

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
0,Super built-up Area,19-Dec,Electronic City Phase II,Coomee,1056.0,2.0,1.0,39.07,2.0,2.0 BHK property in Electronic City Phase II w...
1,Plot Area,Ready To Move,Chikka Tirupathi,Theanmp,2600.0,5.0,3.0,120.00,4.0,4.0 BHK property in Chikka Tirupathi with 2600...
2,Super built-up Area,Ready To Move,Lingadheeranahalli,Soiewre,1521.0,3.0,1.0,95.00,3.0,3.0 BHK property in Lingadheeranahalli with 15...
3,Super built-up Area,Ready To Move,Whitefield,DuenaTa,1170.0,2.0,1.0,38.00,2.0,2.0 BHK property in Whitefield with 1170.0 sqf...
4,Plot Area,Ready To Move,Whitefield,Prrry M,2785.0,5.0,3.0,295.00,4.0,4.0 BHK property in Whitefield with 2785.0 sqf...


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7119 entries, 0 to 7143
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7119 non-null   object 
 1   availability  7119 non-null   object 
 2   location      7119 non-null   object 
 3   society       7119 non-null   object 
 4   total_sqft    7119 non-null   float64
 5   bath          7119 non-null   float64
 6   balcony       7119 non-null   float64
 7   price         7119 non-null   float64
 8   bhk           7119 non-null   float64
 9   text          7119 non-null   object 
dtypes: float64(5), object(5)
memory usage: 611.8+ KB


In [16]:
# find the maxumum length of the text column
max_length = df['text'].apply(len).max()
print("Maximum length of text column:", max_length)

Maximum length of text column: 197


In [17]:
# Creating embedding for each row in the text column using Hugging Face's sentence-transformers library

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss 


# Prepare texts and IDs 

texts = df['text'].tolist()
ids = np.arange(len(texts)).astype('int64')

# Create embeddings 

embed_model = SentenceTransformer('all-MiniLM-L6-v2') # small and fast for generating embeddings

embeddings = embed_model.encode(texts,show_progress_bar=True, convert_to_numpy=True)

# Normalize the embeddings (This helps with the cosine similarity search)

faiss.normalize_L2(embeddings)

# Build Faiss index 

d = embeddings.shape[1] # dimension of the embeddings
cpu_index = faiss.IndexFlatIP(d) # inner product on normalized vectors is equivalent to cosine similarity

# move to gpu 
res = faiss.StandardGpuResources() # Initialize GPU resources
index = faiss.index_cpu_to_gpu(res, 0, cpu_index) # move the index to GPU


index.add(embeddings) # add the embeddings to the index

print(f"Index is on GPU: {index.is_trained} and has {index.ntotal} vectors.")

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 339.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 223/223 [01:15<00:00,  2.95it/s]


Index is on GPU: True and has 7119 vectors.


In [18]:
# Save artifacts to disk to avoid regenerating embeddings each run
import pickle

# 1. Save embeddings (numpy array) - useful for rebuilding or analysis
with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

# 2. Save the FAISS index - this is what you actually search
# Note: GPU index needs to be moved back to CPU before saving
cpu_index_to_save = faiss.index_gpu_to_cpu(index)
faiss.write_index(cpu_index_to_save, "faiss_index.bin")

# 3. Save the texts list - needed to map index IDs back to property descriptions
with open("texts.pkl", "wb") as f:
    pickle.dump(texts, f)

print("Saved: embeddings.pkl, faiss_index.bin, texts.pkl")

Saved: embeddings.pkl, faiss_index.bin, texts.pkl


**Loading Saved Artifacts (Alternative to regenerating embeddings)**

If you've already saved the index and want to skip the embedding generation step, uncomment and run the cell below instead of the embedding generation cell.

In [17]:
# # Uncomment to LOAD saved artifacts instead of regenerating
# import pickle
# import faiss
# from sentence_transformers import SentenceTransformer

# # Load the texts
# with open("texts.pkl", "rb") as f:
#     texts = pickle.load(f)

# # Load embeddings (optional - only needed if you want to inspect them)
# with open("embeddings.pkl", "rb") as f:
#     embeddings = pickle.load(f)

# # Load the FAISS index
# cpu_index = faiss.read_index("faiss_index.bin")

# # Move to GPU
# res = faiss.StandardGpuResources()
# index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# # Load the embedding model (still needed for encoding queries)
# embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# print(f"Loaded index with {index.ntotal} vectors from disk.")

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 573.01it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded index with 7119 vectors from disk.


In [21]:
# Retrival Function

def retrieve(query , top_k=5):
    q_emb = embed_model.encode([query] , convert_to_numpy=True) 
    faiss.normalize_L2(q_emb) # normalize the query embedding
    D, I = index.search(q_emb , top_k) # search the index for the top_k most similar vectors
    results = []

    for idx , score in zip(I[0],D[0]):
        if idx == -1: # if no more results
            continue 
        results.append({'id':int(idx) ,
                         'score' : float(score) ,
                           'text' : texts[int(idx)]
                           }
                      )
        
    return results

In [22]:
# Filter Extractor 

def extract_filters_simple(query):
    query = query.lower()
    filters = {}

    # Extract BHK
    m = re.search(r"(\d+)\s*(bhk|bedroom)" , query)
    if m:
        filters['bhk'] = int(m.group(1))

    # Price
    m = re.search(r'(under|less than|below)\s*(\d+\.?\d*)\s*(lakh|crore)?' , query)
    if m:
        val = float(m.group(2))
        unit = m.group(3)
        if unit and 'crore' in unit:
            val *= 100 # convert crore to lakh
        filters['max_price'] = val
    
    # Sqft

    m = re.search(r'(\d+)\s*(sqft|square)',query)
    if m:
        filters['min_sqft'] = float(m.group(1))
    
    # Location 

    for loc in df['location'].unique():
        if isinstance(loc,str) and loc.lower() in query:
            filters['location'] = loc
            break
    
    return filters



In [23]:
def filter_retrieved(retrieved , df , filters):

    valid = []

    for r in retrieved:
        row = df.iloc[r['id']]

        if 'bhk' in filters and row['bhk'] != filters['bhk']:
            continue
        if 'max_price' in filters and row['price'] > filters['max_price']:
            continue
        if 'min_sqft' in filters and row['total_sqft'] < filters['min_sqft']:
            continue
        if 'location' in filters and row['location'] != filters['location']:
            continue

        valid.append(r)
    
    return valid

In [24]:
import os 
os.environ['HF_HOME'] = "D:/hf_cache"

In [26]:
# Simple response generation using retrieved context : text generation model - Qwen/Qwen2.5-1.5B-Instruct 
from transformers import pipeline 

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
# tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline('text-generation' ,
               model =  model_id ,
                 device = 0 ,
                 dtype = 'auto',
                 model_kwargs = {'cache_dir': "D:/hf_cache"})


def answer_query(query , top_k = 5 , max_new_tokens = 300):


    retrieved = retrieve(query , top_k = top_k)
    
    filters = extract_filters_simple(query)

    valid = filter_retrieved(retrieved , df , filters)

    # choose context

    if len(valid) > 0:
        context_list = valid 
    else:
        context_list = retrieved 


    # Build a concise context string from top results
    context = "\n\n".join([f"Listing {r['id']} : {r['text']}" for r in context_list])

    prompt = f"""You are a real estate assistant.

    Use ONLY the given listings.

    If listings satisfy user query, recommend them.
    If none satisfy, say "No exact matches found".

    Context:
    {context}

    User Query: {query}

    Answer format:
    - Match Summary:
    - Recommended Listings:
    """

    out = gen(prompt , max_new_tokens = max_new_tokens , do_sample = False , return_full_text = False)
    # return full text = False ensures we only get the generated answer without the prompt
    return {'response':out[0]['generated_text'].strip() , 'retrieved':retrieved , 'valid':valid}

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 672.83it/s, Materializing param=model.norm.weight]                              


Other prompt option:


prompt = f"""You are a helpful real estate assistant.
IMPORTANT RULES:

Answer ONLY using the provided listings
If no listings match the user's criteria exactly, say so clearly
Highlight price/size mismatches if they occur
Be concise and factual
Retrieved Properties:
{context}

User Query: {query}

Answer:"""


In [27]:
# Example queries and display result

example_queries = [
    "Show me 3 BHK properties in Whitefield under 80 lakhs",
    "Find 2 BHK apartments in Koramangala with at least 1000 sqft area and price under 1.2 crore",
    "List affordable 1 BHK options in Jayanagar under 50 lakhs"
]

def pretty_print_retrieved(retrieved , df , show_full_row = False):
    print("\nTop retrieved listings:")
    for r in retrieved:
        print(f"  id={r['id']}  score={r['score']:.4f}")
        print(f"  snippet: {r['text']}")
    if show_full_row and len(retrieved)>0:
        print("\nFull dataframe rows for retrieved listings:")
        ids = [r['id'] for r in retrieved]
        display(df.iloc[ids]) # This renders a table in jupyter notebook



In [28]:
for q in example_queries:
    print("\n" + "="*80)
    print("Query:",q)
    try:
        res = answer_query(q,top_k = 5 , max_new_tokens =200)
    except Exception as e:
        print("Error running answer_query:",e)
        continue
    print("\n--- Generated Response ---")
    print(res['response'])
    pretty_print_retrieved(res['retrieved'] , df , show_full_row = False)


Query: Show me 3 BHK properties in Whitefield under 80 lakhs


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated Response ---
No exact matches found. Based on your request for 3 BHK properties in Whitefield under 80 lakhs, I have reviewed the provided listings and found no matching properties that meet both criteria. The closest listing is from Listing 4602 which has a price of 65.0 lakh but does not match the specific requirement of being exactly 3 BHK or within the budget limit. Therefore, no exact matches were found. However, if you're interested in other options, please let me know! 

Please note that while these listings do not perfectly match your query, they may still be suitable depending on your preferences. Let me know how else I can assist you today!

Top retrieved listings:
  id=4442  score=0.7384
  snippet: 3.0 BHK property in Whitefield with 3450.0 sqft, priced at 208.0 lakh. Bathrooms: 4. Balconies: 2. Area type: Plot  Area. Availability: Ready To Move. Society: StodsWa
  id=4999  score=0.7351
  snippet: 3.0 BHK property in Whitefield with 1200.0 sqft, priced at 56.7

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated Response ---
No exact matches found. Based on your criteria of finding 2 BHK apartments in Koramangala with an area of at least 1000 sqft and a price under 1.2 crores, I have reviewed the provided listings. Unfortunately, no listings match all these specific requirements exactly. However, there is one listing that comes close to meeting some of your criteria:

    **Match Summary:** 
    - The listing for 2.0 BHK property in Koramangala with 1300.0 sqft, priced at 82.0 lakh satisfies the condition of having a 2 BHK apartment, being in Koramangala, and having an area of at least 1000 sqft. It also has 2 bathrooms and 2 balconies, which aligns well with your preferences.

    **Recommended Listings:**
    - Listing 3174: 2.0 BHK property in Koram

Top retrieved listings:
  id=3174  score=0.7867
  snippet: 2.0 BHK property in Koramangala with 1300.0 sqft, priced at 82.0 lakh. Bathrooms: 2. Balconies: 2. Area type: Super built-up  Area. Availability: Ready To Move. Society: 

The model is Hallucinating too much.